# Haiku Perturbation Analysis (Metadata-Only) — ohw-31570

**Project Name:** Haiku

## Purpose
- Publication-ready notebook for reproducible evaluation.
- Source patient: **ohw-31570** — Breast cancer, Luminal-A IHC
  (ER+++ 90% / PR+++ 80% / HER2 0 / Ki67 +,20%), T2N0M0/grade 2/IIA.
  Replaces hjv-91786 (region 138) which leaked into the training set.
- Perturbation: **T2N0M0/grade 2/IIA → T4N2M1/grade 3/IV** + Ki67 20% → 45%
  (other clinical metadata unchanged). Then re-run fused HE+text retrieval
  vs the full CODEX corpus and contrast biomarker means top-50 retrievals.

## Notes
- Paths and checkpoints may be environment-specific.
- Run cells top-to-bottom and set your config paths first.


In [ ]:
import sys
from pathlib import Path

# Notebook lives at <repo>/downstream/ — HAIKU_ROOT points at the repo root.
HAIKU_ROOT = Path.cwd().parent
if str(HAIKU_ROOT / 'src') not in sys.path:
    sys.path.append(str(HAIKU_ROOT / 'src'))

# -------- Fill these in to point at your local copies --------
EMBEDDINGS_DIR     = Path('<PATH_TO_PRECOMPUTED_EMBEDDINGS>')
METADATA_DIR       = Path('<PATH_TO_REGION_METADATA>')
BIOMARKER_LIST     = Path('<PATH_TO_BIOMARKER_LIST_PKL>')
ESM_EMBEDDINGS_DIR = Path('<PATH_TO_ESM_EMBEDDINGS>')
CODEX_DATA_DIR     = Path('<PATH_TO_CODEX_INDIVIDUAL_SAMPLES>')
BIOMARKER_LABELS_DIR = Path('<PATH_TO_BIOMARKER_LABELS_DIR>')
SAMPLES_JSON       = HAIKU_ROOT / 'overlap_samples_final.json'
TEST_REGIONS_TXT   = Path('<PATH_TO_test_regions.txt>')
OUTPUT_DIR         = HAIKU_ROOT / 'downstream' / 'figs' / 'perturbation_tnbc_metadata'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from haiku.notebook_utils import setup_notebook, seed_everything
setup_notebook(project_root=str(HAIKU_ROOT))
seed_everything(42)


In [ ]:
import hydra
from omegaconf import DictConfig, OmegaConf
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
import pickle
from os.path import join
import pandas as pd

import numpy as np
import torch.nn.functional as F
import random
import json





from models import Haiku
from data import custom_collate_fn_trimodal, TrimodalDatasetViT
from utils import PerChannelSelfStandardization, CustomGaussianBlurTorch
from datetime import timedelta, datetime
import csv

import warnings
warnings.filterwarnings("ignore")

In [ ]:
he_embedding = torch.load(EMBEDDINGS_DIR / 'he_embedding.pt')
codex_embedding = torch.load(EMBEDDINGS_DIR / 'codex_embedding.pt')
region_label = torch.load(EMBEDDINGS_DIR / 'region_label.pt')
virtual_codex_embedding = torch.load(EMBEDDINGS_DIR / 'virtual_codex_embedding.pt')
text_embedding = torch.load(EMBEDDINGS_DIR / 'text_embedding.pt')
musk_he_embedding = torch.load(EMBEDDINGS_DIR / 'baseline_musk_he_embedding.pt')
musk_codex_embedding = torch.load(EMBEDDINGS_DIR / 'baseline_musk_codex_embedding.pt')
musk_text_embedding = torch.load(EMBEDDINGS_DIR / 'baseline_musk_text_embedding.pt')


In [ ]:
import json
import pandas as pd


sample_dict = json.load(open(SAMPLES_JSON))
sample_ids = list(sample_dict.keys())


with open(TEST_REGIONS_TXT, 'r') as f:
    test_ids = [line.strip() for line in f if line.strip()]

sample_ids = list(set(test_ids) & set(sample_ids))

ref_ids = sorted(sample_ids)


import os
from tqdm import tqdm

region_metadata_dir = str(METADATA_DIR)

# First, read all region_metadata CSVs into a dict: {region_id: df}
region_metadata = {}
metadata_files = [fname for fname in os.listdir(region_metadata_dir) if fname.endswith('.metadata.csv')]
print(f"Reading {len(metadata_files)} region metadata CSVs...")
for fname in tqdm(metadata_files, desc="Reading region metadata", total=len(metadata_files)):
    region_id = fname.split('.')[0]
    try:
        df = pd.read_csv(os.path.join(region_metadata_dir, fname))
        region_metadata[region_id] = df
    except Exception as e:
        print(f"Error reading {fname}: {e}")


In [ ]:
region_metadata_dir = str(METADATA_DIR)

# Initialize dict for filtered DataFrames
bc_tma_metadata = {}

description = {}

grade = {}

stage = {}

metadata_files = [fname for fname in os.listdir(region_metadata_dir) if fname.endswith('.metadata.csv')]
print(f"Reading {len(metadata_files)} region metadata CSVs...")
for fname in tqdm(metadata_files, desc="Reading region metadata", total=len(metadata_files)):
    region_id = fname.split('.')[0]
    try:
        df = pd.read_csv(os.path.join(region_metadata_dir, fname))
        # Make columns lower for searching to be case-insensitive
        columns_lc = [c.lower() for c in df.iloc[:, 1].fillna("").astype(str)]
        values_lc = df.iloc[:, 2].astype(str).str.lower().fillna("")
        # Find rows for tma_description and disease
        tma_descr_idx = [i for i, col in enumerate(columns_lc) if "tma_description" in col]
        disease_idx = [i for i, col in enumerate(columns_lc) if "disease" in col]
        # Check existence
        if tma_descr_idx and disease_idx:
            tma_descr_val = df.iloc[tma_descr_idx[0], 2]
            disease_val = df.iloc[disease_idx[0], 2]
            # Apply filter: tma_description is not nan and disease contains 'breast cancer'
            if pd.notnull(tma_descr_val) and str(tma_descr_val).strip() != "" and "breast cancer" in str(disease_val).lower():
                # Save
                bc_tma_metadata[region_id] = df
                description[region_id] = tma_descr_val
                grade[region_id] = df[df['FEATURE_NAME'] == 'grade']['FEATURE_VALUE'].iloc[0]
                stage[region_id] = df[df['FEATURE_NAME'] == 'stage']['FEATURE_VALUE'].iloc[0]
    except Exception as e:
        print(f"Error reading {fname}: {e}")


In [ ]:
# paired_fusion_retrieval.py
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

# -------------------------
# Metrics from score matrix
# -------------------------
@torch.no_grad()
def _compute_ranked_metrics_from_sorted_match(sorted_match: torch.Tensor, top_ks):
    device = sorted_match.device
    B, N = sorted_match.shape
    sm = sorted_match.float()
    has_match = sm.bool().any(dim=1)

    cumsum_rel = torch.cumsum(sm, dim=1)
    ranks = torch.arange(1, N + 1, device=device).float().unsqueeze(0)

    first_hit_pos = sm.bool().float().argmax(dim=1) + 1
    first_hit_pos = torch.where(has_match, first_hit_pos, torch.zeros_like(first_hit_pos))
    mrr = torch.where(has_match, 1.0 / first_hit_pos.clamp(min=1).float(),
                      torch.zeros_like(first_hit_pos, dtype=torch.float))

    precision_at_i = cumsum_rel / ranks
    precision_at_relevant = precision_at_i * sm
    num_rel = sm.sum(dim=1)
    ap_full = torch.where(num_rel > 0,
                          precision_at_relevant.sum(dim=1) / num_rel,
                          torch.zeros_like(num_rel))

    acc = {
        "topk_hits": {k: 0 for k in top_ks},
        "prec_at_k_sum": {k: 0.0 for k in top_ks},
        "rec_at_k_sum":  {k: 0.0 for k in top_ks},
        "f1_at_k_sum":   {k: 0.0 for k in top_ks},
        "ndcg_at_k_sum": {k: 0.0 for k in top_ks},
        "map_at_k_sum":  {k: 0.0 for k in top_ks},
        "ranks_first_list": [],
        "rr_list": [],
        "ap_list": [],
    }
    if has_match.any(): acc["ranks_first_list"].append(first_hit_pos[has_match])
    acc["rr_list"].append(mrr)
    acc["ap_list"].append(ap_full)

    for k in top_ks:
        topk_rel = sm[:, :k]
        hits = topk_rel.bool().any(dim=1).sum().item()
        acc["topk_hits"][k] += hits

        prec_k = topk_rel.sum(dim=1) / float(k)
        rec_k  = torch.where(num_rel > 0, topk_rel.sum(dim=1) / num_rel, torch.zeros_like(num_rel))
        denom  = (prec_k + rec_k).clamp(min=1e-12)
        f1_k   = 2 * (prec_k * rec_k) / denom

        positions = torch.arange(1, k + 1, device=device).float().unsqueeze(0)
        discounts = torch.log2(positions + 1.0)
        dcg = (topk_rel / discounts).sum(dim=1)
        ideal_k = torch.minimum(num_rel, torch.tensor(float(k), device=device))
        inv_log = 1.0 / torch.log2(torch.arange(2, k + 2, device=device).float())
        inv_log_cumsum = torch.cumsum(inv_log, dim=0)
        idcg = torch.where(ideal_k > 0,
                           inv_log_cumsum[(ideal_k.long() - 1).clamp(min=0)],
                           torch.zeros_like(ideal_k))
        ndcg_k = torch.where(idcg > 0, dcg / idcg, torch.zeros_like(dcg))

        precision_at_i_k = precision_at_i[:, :k]
        rel_k = sm[:, :k]
        ap_k = torch.where(num_rel > 0,
                           (precision_at_i_k * rel_k).sum(dim=1) / num_rel,
                           torch.zeros_like(num_rel))

        acc["prec_at_k_sum"][k] += prec_k.sum().item()
        acc["rec_at_k_sum"][k]  += rec_k.sum().item()
        acc["f1_at_k_sum"][k]   += f1_k.sum().item()
        acc["ndcg_at_k_sum"][k] += ndcg_k.sum().item()
        acc["map_at_k_sum"][k]  += ap_k.sum().item()
    return acc

@torch.no_grad()
def _merge_accumulators(acc_list, top_ks, N_query_total):
    metrics = {}
    total_hits = {k: 0 for k in top_ks}
    prec_sum = {k: 0.0 for k in top_ks}
    rec_sum  = {k: 0.0 for k in top_ks}
    f1_sum   = {k: 0.0 for k in top_ks}
    ndcg_sum = {k: 0.0 for k in top_ks}
    mapk_sum = {k: 0.0 for k in top_ks}
    ranks_first, rr_all, ap_all = [], [], []

    for acc in acc_list:
        for k in top_ks:
            total_hits[k] += acc["topk_hits"][k]
            prec_sum[k]   += acc["prec_at_k_sum"][k]
            rec_sum[k]    += acc["rec_at_k_sum"][k]
            f1_sum[k]     += acc["f1_at_k_sum"][k]
            ndcg_sum[k]   += acc["ndcg_at_k_sum"][k]
            mapk_sum[k]   += acc["map_at_k_sum"][k]
        if acc["ranks_first_list"]: ranks_first.append(torch.cat(acc["ranks_first_list"]))
        rr_all.append(torch.cat(acc["rr_list"]))
        ap_all.append(torch.cat(acc["ap_list"]))

    num_queries = float(N_query_total)
    for k in top_ks:
        hit_rate = total_hits[k] / num_queries
        metrics[f"top{k}_acc"] = hit_rate
        metrics[f"CMC@{k}"]    = hit_rate
        metrics[f"P@{k}"]      = prec_sum[k]   / num_queries
        metrics[f"R@{k}"]      = rec_sum[k]    / num_queries
        metrics[f"F1@{k}"]     = f1_sum[k]     / num_queries
        metrics[f"nDCG@{k}"]   = ndcg_sum[k]   / num_queries
        metrics[f"mAP@{k}"]    = mapk_sum[k]   / num_queries

    if ranks_first:
        rf = torch.cat(ranks_first)
        metrics["MR"]   = rf.float().mean().item()
        metrics["MedR"] = rf.median().item()
    else:
        metrics["MR"] = float('nan'); metrics["MedR"] = float('nan')
    metrics["MRR"] = torch.cat(rr_all).mean().item()
    metrics["mAP"] = torch.cat(ap_all).mean().item()
    return metrics

@torch.no_grad()
def _metrics_from_scores(scores: torch.Tensor, match_matrix: torch.Tensor,
                         top_ks=(1,5,10,20,50), batch_size=256, device='cuda', desc="eval"):
    scores = scores.to(device); match_matrix = match_matrix.to(device)
    Nq, Ng = scores.shape; num_batches = (Nq + batch_size - 1) // batch_size
    accs = []
    with tqdm(total=num_batches, desc=desc) as pbar:
        for start in range(0, Nq, batch_size):
            end = min(start + batch_size, Nq)
            batch_scores = scores[start:end]; mm = match_matrix[start:end]
            sorted_idx = torch.argsort(batch_scores, dim=1, descending=True)
            sorted_match = torch.gather(mm.float().squeeze(-1), 1, sorted_idx).bool()
            accs.append(_compute_ranked_metrics_from_sorted_match(sorted_match, top_ks))
            pbar.set_postfix({"Batch": mm.size(0)}); pbar.update(1)
    return _merge_accumulators(accs, top_ks, Nq), sorted_idx

def _build_match_matrix(query_ids, gallery_ids, query_labels, gallery_labels, mode, device):
    if mode == 'exact':
        mm = (np.array(query_ids)[:, None] == np.array(gallery_ids)[None, :])
    else:
        if query_labels is None or gallery_labels is None:
            raise ValueError("mode='label' requires labels.")
        mm = (np.array(query_labels)[:, None] == np.array(gallery_labels)[None, :])
    return torch.from_numpy(mm).to(torch.bool).to(device)

# -------------------------
# Rank fusion utility (RRF)
# -------------------------
@torch.no_grad()
def _scores_to_rrf(he_scores, text_scores, w_he=0.7, w_text=0.3, k=60):
    """
    Reciprocal Rank Fusion with weights: s = w_he/(k + rank_he) + w_text/(k + rank_text).
    Inputs: (B, C) scores where higher is better. Output: (B, C) fused scores.
    """
    # ranks: 1 = best
    order_he   = torch.argsort(he_scores,   dim=1, descending=True)
    order_text = torch.argsort(text_scores, dim=1, descending=True)
    inv_he   = torch.empty_like(order_he);   inv_text = torch.empty_like(order_text)
    ar = torch.arange(he_scores.size(1), device=he_scores.device).unsqueeze(0)
    inv_he.scatter_(1, order_he, ar); inv_text.scatter_(1, order_text, ar)
    ranks_he   = inv_he + 1; ranks_text = inv_text + 1
    return w_he / (k + ranks_he.float()) + w_text / (k + ranks_text.float())

# ----------------------------------------------
# Paired (text, HE) → CODEX fusion & comparison
# ----------------------------------------------
@torch.no_grad()
def run_paired_fusion_compare(
    text_embeddings: torch.Tensor,
    he_embeddings: torch.Tensor,
    codex_embeddings: torch.Tensor,
    # ids/labels for evaluation
    query_ids, codex_ids,
    query_labels=None, codex_labels=None,
    # if you also want HE-as-gallery baseline (optional)
    he_ids=None, he_labels=None,
    mode='label',                 # 'exact' or 'label' for evaluation vs CODEX
    top_ks=(1,5,10,20,50),
    # fusion options
    fusion="wsum",                # "wsum" | "softmax" | "rrf"
    w_he=0.7, w_text=0.3,         # HE is stronger → give it more weight
    tau_he=0.07, tau_text=0.07,   # temps for softmax fusion (if used)
    rrf_k=60,                     # RRF constant (if used)
    # score normalization (helps when scales differ)
    zscore=False,                 # per-query z-score before fusion
    batch_size=256, device='cuda'
):
    """
    Build baselines (text→CODEX, HE→CODEX) and a paired-fusion ranking (text+HE → CODEX).
    Compare metrics for all three.
    Additionally, return the index with the largest fused-top5 improvement over HE, and the improvement value.
    Also, return query indices for which BOTH text and HE hit the *target* index in their top5 (top-5 contains a true relevant item for both text and he).
    """
    assert mode in ['exact','label']
    assert abs(w_he + w_text - 1.0) < 1e-6, "w_he + w_text must sum to 1."

    # Normalize embeddings and move
    text  = F.normalize(text_embeddings, dim=1).to(device)
    he    = F.normalize(he_embeddings,   dim=1).to(device)
    codex = F.normalize(codex_embeddings,dim=1).to(device)

    Nq, C = text.size(0), codex.size(0)

    # Build relevance against CODEX
    mm_qc = _build_match_matrix(query_ids, codex_ids, query_labels, codex_labels, mode, device)

    # Baseline similarities
    s_text2codex = text  @ codex.T   # (Nq, C)
    s_he2codex   = he    @ codex.T   # (H,  C)  <-- but we need HE paired per query!

    # Paired HE for each text query.
    assert he.size(0) == Nq, "Provide HE embeddings paired to each text (same Nq)."
    s_pair_he2codex = he @ codex.T   # (Nq, C)

    # Optionally z-score per query (to align scales before fusion)
    def _zscore_per_row(x):
        mu = x.mean(dim=1, keepdim=True); sd = x.std(dim=1, keepdim=True).clamp_min(1e-6)
        return (x - mu) / sd
    s_t = _zscore_per_row(s_text2codex)      if zscore else s_text2codex
    s_h = _zscore_per_row(s_pair_he2codex)   if zscore else s_pair_he2codex

    # ---------- FUSION ----------
    if fusion == "wsum":
        fused_scores = w_he * s_h + w_text * s_t
    elif fusion == "softmax":
        p_h = torch.softmax(s_h / max(tau_he, 1e-6), dim=1)
        p_t = torch.softmax(s_t / max(tau_text, 1e-6), dim=1)
        fused_scores = w_he * p_h + w_text * p_t
    elif fusion == "rrf":
        fused_scores = _scores_to_rrf(s_h, s_t, w_he=w_he, w_text=w_text, k=rrf_k)
    else:
        raise ValueError(f"Unknown fusion='{fusion}' (use 'wsum' | 'softmax' | 'rrf').")

    # ---------- Evaluate ----------
    print("\n[Baseline] text→CODEX")
    text_baseline, test_index = _metrics_from_scores(s_text2codex, mm_qc, top_ks, batch_size, device, desc="baseline t→C")
    print("\n[Baseline] HE→CODEX (paired HE)")
    he_baseline, he_index   = _metrics_from_scores(s_pair_he2codex, mm_qc, top_ks, batch_size, device, desc="baseline HE→C")
    print(f"\n[Paired Fusion] ({fusion}; w_he={w_he}, w_text={w_text}" +
          (f", τ_he={tau_he}, τ_text={tau_text}" if fusion=='softmax' else "") +
          (f", rrf_k={rrf_k}" if fusion=='rrf' else "") +
          (", zscore" if zscore else "") + ")")
    fused_metrics, fused_index = _metrics_from_scores(fused_scores, mm_qc, top_ks, batch_size, device, desc="fusion(t+HE)→C")

    # ---------- Per-query Top5 comparison ----------
    # Compute, for each query, the number of relevant CODEX entries in the top5 ranked by each method
    def get_topk_recall_per_query(scores, match_matrix, k=5):
        # scores: (Nq, Ng)
        # match_matrix: (Nq, Ng)
        sorted_idx = torch.argsort(scores, dim=1, descending=True)
        topk_idx = sorted_idx[:, :k]
        # gather relevant labels for these top-k
        relevant_topk = torch.gather(match_matrix, 1, topk_idx)
        relevant_topk = relevant_topk.float() # (Nq, k), 1 for relevant, 0 else
        return relevant_topk.sum(dim=1), topk_idx, relevant_topk  # ALSO return which indices

    topk = 10
    # Ensure tensors on proper device
    mm_qc_float = mm_qc.float()

    # Obtain recall, indices, and binary relevant hits in topk for all three methods
    fused_top5_recall, fused_top5_idx, fused_relevant_topk = get_topk_recall_per_query(fused_scores, mm_qc_float, k=topk)
    he_top5_recall, he_top5_idx, he_relevant_topk         = get_topk_recall_per_query(s_pair_he2codex, mm_qc_float, k=topk)
    text_top5_recall, text_top5_idx, text_relevant_topk   = get_topk_recall_per_query(s_text2codex, mm_qc_float, k=topk)

    # Improvement of fusion over HE for each query
    fused_improvement = fused_top5_recall - he_top5_recall

    # Find the query index with the biggest improvement (and value)
    max_improve_value, max_improve_index = torch.max(fused_improvement, dim=0)
    max_improve_index = max_improve_index.item()
    max_improve_value = max_improve_value.item()

    # For reference, also the recall values for this query
    stats_at_max = {
        'fused_top5_recall': fused_top5_recall[max_improve_index].item(),
        'he_top5_recall': he_top5_recall[max_improve_index].item(),
        'text_top5_recall': text_top5_recall[max_improve_index].item()
    }

    # ---------- Also, return query indices where BOTH text and HE hit at least one relevant item in top5 ----------
    # For each query, relevant_topk is of shape (Nq, 5): value 1 if relevant, 0 else in each of 5 slots.
    # For 'hit', check if sum in topk >= 1
    hit_text = (text_relevant_topk.sum(dim=1) >= 1)
    hit_he   = (he_relevant_topk.sum(dim=1) >= 1)
    both_hit_indices = torch.nonzero(hit_text & hit_he).view(-1).cpu().numpy().tolist()

    # ---------- Summary ----------
    def _print_row(k, base, name):
        return f"@{k}  CMC:{base[f'top{k}_acc']:.4f}  P:{base[f'P@{k}']:.4f}  R:{base[f'R@{k}']:.4f}  F1:{base[f'F1@{k}']:.4f}  nDCG:{base[f'nDCG@{k}']:.4f}  mAP@k:{base[f'mAP@{k}']:.4f}"

    print("\n=== Comparison vs CODEX ===")
    for k in top_ks:
        print("[text→C ]", _print_row(k, text_baseline, "text"))
        print("[HE→C   ]", _print_row(k, he_baseline, "he"))
        print("[fusion  ]", _print_row(k, fused_metrics, "fusion"))
        print("-")

    def _print_global(m, tag):
        print(f"{tag}  MR:{m['MR']:.4f}  MedR:{m['MedR']:.4f}  MRR:{m['MRR']:.4f}  mAP:{m['mAP']:.4f}")
    _print_global(text_baseline, "[text→C ]")
    _print_global(he_baseline,   "[HE→C   ]")
    _print_global(fused_metrics, "[fusion  ]")

    print(f"\n[FUSION GAIN] Largest improvement (top5 recall fusion - HE): index={max_improve_index}, Δ={max_improve_value}")
    print(f"  Details: fused={stats_at_max['fused_top5_recall']}  HE={stats_at_max['he_top5_recall']}  text={stats_at_max['text_top5_recall']}")

    ret_dict = {
        "text_baseline": text_baseline,
        "he_baseline": he_baseline,
        "fusion": fused_metrics,
        'fused_index': fused_index,
        'test_index': test_index,
        'he_index': he_index,
        'fusion_top5_gain': {
            'max_improve_index': max_improve_index,
            'max_improve_value': max_improve_value,
            'stats_at_max': stats_at_max
        },
        'both_text_he_top5_hit_indices': both_hit_indices  # <== Added: indices where both text and HE hit relevant in top5
    }

    return ret_dict, fused_scores


In [ ]:
# Original retrieval will be run after model loading (Cell 8)
# using metadata-only text description encoded through the model.
# See cells below for both original and perturbed metadata-only retrievals.
pass


In [ ]:
from models import Haiku
cfg = OmegaConf.load(HAIKU_ROOT / 'src' / 'configs' / 'config.yaml')

from data import custom_collate_fn_trimodal, TrimodalDatasetViT, TrimodalDatasetViTEmbedding, custom_collate_fn_embedding, TrimodalDatasetViTPickeVerion

import warnings
warnings.filterwarnings("ignore")



sample_dict = json.load(open(SAMPLES_JSON))
sample_ids = list(sample_dict.keys())


with open(TEST_REGIONS_TXT, 'r') as f:
    test_ids = [line.strip() for line in f if line.strip()]

sample_ids = list(set(test_ids) & set(sample_ids))

ref_ids = sorted(sample_ids)


he_embedding = torch.load(EMBEDDINGS_DIR / 'he_embedding.pt')
codex_embedding = torch.load(EMBEDDINGS_DIR / 'codex_embedding.pt')
region_label = torch.load(EMBEDDINGS_DIR / 'region_label.pt')
virtual_codex_embedding = torch.load(EMBEDDINGS_DIR / 'virtual_codex_embedding.pt')
text_embedding = torch.load(EMBEDDINGS_DIR / 'text_embedding.pt')
musk_he_embedding = torch.load(EMBEDDINGS_DIR / 'baseline_musk_he_embedding.pt')
musk_codex_embedding = torch.load(EMBEDDINGS_DIR / 'baseline_musk_codex_embedding.pt')
musk_text_embedding = torch.load(EMBEDDINGS_DIR / 'baseline_musk_text_embedding.pt')


vocab = pickle.load(open(BIOMARKER_LIST, 'rb'))

vocab[vocab == 'PGP9.5'] = 'PGP9_5'

for i in range(len(vocab)):
    if '.' in vocab[i]:
        vocab[i] = vocab[i].replace('.', '_')

cfg.model.vocab = vocab

# Load pretrained Haiku model directly from HuggingFace
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tokenizer, marker_embedding = Haiku.from_pretrained("zhihuanglab/Haiku", device=device)

cfg.dataset.codex_path = str(CODEX_DATA_DIR)



In [ ]:
# Metadata-only descriptions: no biomarker/molecular profile info
# Source patient: ohw-31570 (Breast Cancer, Luminal-A IHC)
# Original: T2N0M0 / grade 2 / IIA / Ki-67 +,20%
# Perturbed: T4N2M1 / grade 3 / IV  / Ki-67 +,45%
# (Replaces hjv-91786 from the leakage-affected version of this notebook.)

original_metadata_description = (
    "A representative section of Breast tissue highlighting the landscape of Breast Cancer. "
    "Additional clinical details include: age: 44, sex: F, tissue_type: Breast, "
    "disease: Breast Cancer, diagnosis: Invasive carcinoma of no special type, "
    "Pathology diagnosis: Invasive carcinoma of no special type, tnm: T2N0M0, grade: 2, "
    "stage: IIA, type: Malignant, "
    "tma_description: Breast cancer TMA with survival data and IHC results, "
    "IHC ER: +++,90%, IHC PR: +++,80%, IHC HER2: 0, IHC Ki67: +,20%."
)

perturbed_metadata_description = (
    "A representative section of Breast tissue highlighting the landscape of Breast Cancer. "
    "Additional clinical details include: age: 44, sex: F, tissue_type: Breast, "
    "disease: Breast Cancer, diagnosis: Invasive carcinoma of no special type, "
    "Pathology diagnosis: Invasive carcinoma of no special type, tnm: T4N2M1, grade: 3, "
    "stage: IV, type: Malignant, "
    "tma_description: Breast cancer TMA with survival data and IHC results, "
    "IHC ER: +++,90%, IHC PR: +++,80%, IHC HER2: 0, IHC Ki67: +,45%."
)

# Encode both descriptions through the model
input_texts = [original_metadata_description, perturbed_metadata_description]

text_preprocess = sample_data.text_processing

text_query = {'text': [], 'att_mask': []}

for text in input_texts:
    input_ids, attention_mask = text_preprocess(text)
    text_query['text'].append(input_ids)
    text_query['att_mask'].append(attention_mask)

text_query['text'] = torch.stack(text_query['text'])
text_query['att_mask'] = torch.stack(text_query['att_mask'])

model.eval()

new_text_embedding = model.get_features_single_modality(text_query, modality='text')

print(f"Original metadata-only embedding shape: {new_text_embedding[0].shape}")
print(f"Perturbed metadata-only embedding shape: {new_text_embedding[1].shape}")


In [ ]:
check_region = 230  # ohw-31570

query_ids = torch.arange(codex_embedding.shape[0])
codex_ids = torch.arange(codex_embedding.shape[0])

# --- Original metadata-only retrieval ---
original_text_embedding = text_embedding[(region_label==check_region).reshape(-1)]
original_text_embedding = new_text_embedding[0].repeat(original_text_embedding.shape[0], 1)

print("=== Original (metadata-only) retrieval ===")
metrics, fused_scores = run_paired_fusion_compare(
        original_text_embedding, he_embedding[(region_label==check_region).reshape(-1)], codex_embedding,
        query_ids=query_ids[(region_label==check_region).reshape(-1)], codex_ids=codex_ids,
        query_labels=region_label[(region_label==check_region).reshape(-1)], codex_labels=region_label,
        mode='exact',
        top_ks=(1,5,10,20,50),
        fusion="wsum",
        w_he=0.6, w_text=0.4,
        tau_he=0.07, tau_text=0.07,
        rrf_k=60,
        zscore=False,
        batch_size=1024,
        device='cpu'
)

# --- Perturbed metadata-only retrieval ---
perturbed_text_embedding = text_embedding[(region_label==check_region).reshape(-1)]
perturbed_text_embedding = new_text_embedding[1].repeat(perturbed_text_embedding.shape[0], 1)

print("\n=== Perturbed (metadata-only) retrieval ===")
late_metrics, late_fused_scores = run_paired_fusion_compare(
        perturbed_text_embedding, he_embedding[(region_label==check_region).reshape(-1)], codex_embedding,
        query_ids=query_ids[(region_label==check_region).reshape(-1)], codex_ids=codex_ids,
        query_labels=region_label[(region_label==check_region).reshape(-1)], codex_labels=region_label,
        mode='exact',
        top_ks=(1,5,10,20,50),
        fusion="wsum",
        w_he=0.6, w_text=0.4,
        tau_he=0.07, tau_text=0.07,
        rrf_k=60,
        zscore=False,
        batch_size=1024,
        device='cpu'
)


In [ ]:
import pickle
import numpy as np
from tqdm import tqdm

late_index_fused = late_metrics['fused_index']
late_top_index_fused = late_metrics['both_text_he_top5_hit_indices']

scores = late_fused_scores

late_biomarker_weighted_means = []
late_corresponding_patches = {'codex': [], 'channels': [], 'HE': [], 'region_id': [], 'k': []}

description_json_dir = str(BIOMARKER_LABELS_DIR) + '/'

for i, index in tqdm(enumerate(late_index_fused.tolist()), desc="Processing corresponding patches", total=len(late_index_fused)):
    late_biomarker_label = {}
    index_codex = []
    index_channels = []
    index_HE = []
    index_region = []
    index_k = []

    k_indices = index[:50]
    region_ids = [sample_data[k]['region_id'] for k in k_indices]
    patch_ids = [sample_data[k]['patch_id'] for k in k_indices]

    # Get the scores for these k_indices (assume `scores` is [n_queries, n_results])
    # Here, we use direct scores as weights (no normalization).
    k_scores = np.array([scores[i, k] for k in k_indices])
    k_weights = k_scores  # do NOT normalize

    # Load bm values for all top patches and aggregate into dictionary
    bm_per_patch = [{} for _ in range(len(k_indices))]
    for idx, (k, region_id, patch_id) in enumerate(zip(k_indices, region_ids, patch_ids)):
        index_region.append(region_id)
        index_k.append(k)
        with open(description_json_dir + region_id + f'/{patch_id}_backgroud.json', 'r') as f:
            description = json.load(f)
        for bm, v in description.items():
            if '.' in bm:
                bm = bm.replace('.', '_')
            bm_per_patch[idx][bm] = v

    # Convert values for each biomarker into lists for weighted mean
    # Collect values for each biomarker across all patches
    bm_val_lists = {}
    for bm_idx in range(len(bm_per_patch)):
        for bm, val in bm_per_patch[bm_idx].items():
            if bm not in bm_val_lists:
                bm_val_lists[bm] = []
            bm_val_lists[bm].append((val, bm_idx))

    bm_weighted_means = {}
    for bm, val_idx_list in bm_val_lists.items():
        values = []
        patch_weights = []
        for v, idx in val_idx_list:
            try:
                v_f = float(v)
                values.append(v_f)
                patch_weights.append(k_weights[idx])
            except Exception:
                continue
        if len(values) > 0 and np.sum(patch_weights) > 0:
            weighted_mean = float(np.average(values, weights=patch_weights))
            bm_weighted_means[bm] = weighted_mean
        else:
            bm_weighted_means[bm] = None

    late_biomarker_weighted_means.append(bm_weighted_means)
    late_corresponding_patches['region_id'].append(index_region)
    late_corresponding_patches['k'].append(index_k)

with open('late_biomarker_means.pkl', 'wb') as f:
    pickle.dump(late_biomarker_weighted_means, f)

with open('late_corresponding_patches.pkl', 'wb') as f:
    pickle.dump(late_corresponding_patches, f)


In [ ]:
import pickle
import numpy as np
from tqdm import tqdm

late_index_fused = metrics['fused_index']
late_top_index_fused = metrics['both_text_he_top5_hit_indices']

scores = fused_scores

late_biomarker_weighted_means = []
late_corresponding_patches = {'codex': [], 'channels': [], 'HE': [], 'region_id': [], 'k': []}

description_json_dir = str(BIOMARKER_LABELS_DIR) + '/'

for i, index in tqdm(enumerate(late_index_fused.tolist()), desc="Processing corresponding patches", total=len(late_index_fused)):
    late_biomarker_label = {}
    index_codex = []
    index_channels = []
    index_HE = []
    index_region = []
    index_k = []

    k_indices = index[:50]
    region_ids = [sample_data[k]['region_id'] for k in k_indices]
    patch_ids = [sample_data[k]['patch_id'] for k in k_indices]

    # Get the scores for these k_indices (assume `scores` is [n_queries, n_results])
    # Here, we use direct scores as weights (no normalization).
    k_scores = np.array([scores[i, k] for k in k_indices])
    k_weights = k_scores  # do NOT normalize

    # Load bm values for all top patches and aggregate into dictionary
    bm_per_patch = [{} for _ in range(len(k_indices))]
    for idx, (k, region_id, patch_id) in enumerate(zip(k_indices, region_ids, patch_ids)):
        index_region.append(region_id)
        index_k.append(k)
        with open(description_json_dir + region_id + f'/{patch_id}_backgroud.json', 'r') as f:
            description = json.load(f)
        for bm, v in description.items():
            if '.' in bm:
                bm = bm.replace('.', '_')
            bm_per_patch[idx][bm] = v

    # Convert values for each biomarker into lists for weighted mean
    # Collect values for each biomarker across all patches
    bm_val_lists = {}
    for bm_idx in range(len(bm_per_patch)):
        for bm, val in bm_per_patch[bm_idx].items():
            if bm not in bm_val_lists:
                bm_val_lists[bm] = []
            bm_val_lists[bm].append((val, bm_idx))

    bm_weighted_means = {}
    for bm, val_idx_list in bm_val_lists.items():
        values = []
        patch_weights = []
        for v, idx in val_idx_list:
            try:
                v_f = float(v)
                values.append(v_f)
                patch_weights.append(k_weights[idx])
            except Exception:
                continue
        if len(values) > 0 and np.sum(patch_weights) > 0:
            weighted_mean = float(np.average(values, weights=patch_weights))
            bm_weighted_means[bm] = weighted_mean
        else:
            bm_weighted_means[bm] = None

    late_biomarker_weighted_means.append(bm_weighted_means)
    late_corresponding_patches['region_id'].append(index_region)
    late_corresponding_patches['k'].append(index_k)

with open('mid_biomarker_means.pkl', 'wb') as f:
    pickle.dump(late_biomarker_weighted_means, f)

with open('mid_corresponding_patches.pkl', 'wb') as f:
    pickle.dump(late_corresponding_patches, f)


In [ ]:
query_patches = {'HE':[], 'codex':[], 'channels':[], 'text':[], 'patch_id':[]}

check_region = 230  # ohw-31570

for i in torch.arange(codex_embedding.shape[0])[(region_label==check_region).reshape(-1)].numpy().tolist():
    query_patches['HE'].append(sample_data[i]['HandE'])
    query_patches['codex'].append(sample_data[i]['codex'])
    query_patches['channels'].append(sample_data[i]['channels'])
    query_patches['text'].append(sample_data[i]['raw_text'])
    query_patches['patch_id'].append(sample_data[i]['patch_id'])

In [ ]:
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 15
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 20
plt.rcParams['ytick.labelsize'] = 20
plt.rcParams['axes.linewidth'] = 2  # Increase boundary (axes/frame) line width

# --------- SECTION 1: Get KMeans cluster assignments on the source region's H&E embeddings ---------
if isinstance(he_embedding, np.ndarray):
    he_emb_np = he_embedding[(region_label==230).reshape(-1)]
else:
    he_emb_np = he_embedding.detach().cpu().numpy()[(region_label==230).reshape(-1)]

num_clusters = 4
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
he_labels = kmeans.fit_predict(he_emb_np)
he_prototypes = kmeans.cluster_centers_

# --------- SECTION 1.5: Visualize t-SNE for these embeddings ---------
print("Running t-SNE for region ohw-31570 H&E embeddings...")
tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto', perplexity=20)
he_emb_tsne = tsne.fit_transform(he_emb_np)

fig, ax = plt.subplots(figsize=(7, 6))
cmap = plt.get_cmap("tab10")

# Prepare colors so both scatter and legend match exactly
sample_colors = [cmap(lbl) for lbl in he_labels]
scatter = ax.scatter(he_emb_tsne[:, 0], he_emb_tsne[:, 1], c=sample_colors, s=60, alpha=0.8)

#ax.set_title("t-SNE of ohw-31570 H&E Embeddings (colored by KMeans cluster)", fontsize=16)
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
# Remove x and y ticks
ax.set_xticks([])
ax.set_yticks([])

# Ensure legend matches colors in scatter & draw legend here
category_handles = [
    Line2D([0], [0], marker='o', color='w', label=f'Cluster {i}',
           markerfacecolor=cmap(i), markersize=10)
    for i in range(num_clusters)
]
legend = ax.legend(handles=category_handles, title="KMeans Cluster", loc='best', frameon=True)
ax.add_artist(legend)

plt.tight_layout()
#os.makedirs(str(OUTPUT_DIR), exist_ok=True)
#plt.savefig(str(OUTPUT_DIR / "tsne_region138_kmeans.svg"), bbox_inches="tight")
#plt.savefig(str(OUTPUT_DIR / "tsne_region138_kmeans.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# Save intermediate results for drawing notebooks
# ============================================================
import pickle, os

save_dir = str(OUTPUT_DIR)
os.makedirs(save_dir, exist_ok=True)

intermediate = {
    'he_emb_np': he_emb_np,
    'he_labels': he_labels,
    'he_prototypes': he_prototypes,
    'he_emb_tsne': he_emb_tsne,
    'query_patches': query_patches,
    'check_region': check_region,
    'num_clusters': num_clusters,
}

pkl_path = os.path.join(save_dir, 'tnbc_intermediate.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(intermediate, f)
print(f'Saved intermediate results to {pkl_path}')
print(f'  he_labels: {he_labels.shape}, he_emb_tsne: {he_emb_tsne.shape}')
print(f'  query_patches HE count: {len(query_patches["HE"])}')


In [ ]:

from PIL import Image
import os

def _to_hwc_rgb01(he):
    x = np.asarray(he)
    assert x.ndim == 3 and (x.shape[0] == 3 or x.shape[-1] == 3), f"HE must be (3,H,W) or (H,W,3); got {x.shape}"
    if x.shape[0] == 3:
        x = np.moveaxis(x, 0, -1)
    if x.dtype != np.uint8:
        mn, mx = float(x.min()), float(x.max())
        if mx > mn:
            x = (x - mn) / (mx - mn) * 255.0
        x = x.astype(np.uint8)
    pil = Image.fromarray(x).convert("RGB")
    return np.asarray(pil, dtype=np.float32) / 255.0  # (H,W,3) float[0,1]

# --------- SECTION 2+3: For each cluster, save three closest prototypes as RGB in separate panel images (one file per cluster) ---------
for cluster_idx in range(num_clusters):
    cluster_member_indices = np.where(he_labels == cluster_idx)[0]
    if len(cluster_member_indices) == 0:
        print(f"Warning: No samples found for cluster {cluster_idx}.")
        continue

    prototypes_dists = np.linalg.norm(
        he_emb_np[cluster_member_indices] - he_prototypes[cluster_idx], axis=1
    )
    # Indices of the three closest samples to the prototype (centroid)
    n_show = min(3, len(cluster_member_indices))
    sorted_within_cluster = np.argsort(prototypes_dists)[:n_show]
    prototype_indices_global = cluster_member_indices[sorted_within_cluster]

    # Collect RGB images for each prototype for this cluster
    rgb_imgs = []
    for i, proto_idx_global in enumerate(prototype_indices_global):
        try:
            prototype_he_patch = query_patches['HE'][proto_idx_global]
            if prototype_he_patch.ndim == 3:
                img_rgb = _to_hwc_rgb01(prototype_he_patch)
                rgb_imgs.append(img_rgb)
            else:
                print(f"HE patch for prototype {i+1} in cluster {cluster_idx} is not 3D.")
                rgb_imgs.append(None)
        except Exception as e:
            print(f"Failed to process prototype {i+1} for cluster {cluster_idx}: {e}")
            rgb_imgs.append(None)

    # Panel: plot and save a single SVG file per cluster showing the closest prototypes
    save_dir = "region138_cluster_prototype_HE_panels"
    os.makedirs(save_dir, exist_ok=True)
    panel_fname = os.path.join(save_dir, f"region138_cluster{cluster_idx}_prototypes_panel.svg")

    plt.figure(figsize=(4*n_show, 5))
    for i, img_rgb in enumerate(rgb_imgs):
        plt.subplot(1, n_show, i + 1)
        if img_rgb is not None:
            plt.imshow(img_rgb)
            plt.title(f"Prototype #{i+1}")
        else:
            plt.title(f"Prototype #{i+1} (missing)")
        plt.axis('off')
    plt.suptitle(f"Cluster {cluster_idx}: Top {n_show} Prototype HE Patches (Region ohw-31570)")
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    # Save this panel image as SVG
    plt.savefig(str(OUTPUT_DIR / f"prototype_HE_cluster_{cluster_idx}.svg"), bbox_inches="tight")
    plt.savefig(str(OUTPUT_DIR / f"prototype_HE_cluster_{cluster_idx}.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # Additionally, optionally save individual SVGs for completeness in a subdir (comment out if not needed)
    indv_dir = os.path.join(save_dir, f"cluster{cluster_idx}_individuals")
    os.makedirs(indv_dir, exist_ok=True)
    for i, img_rgb in enumerate(rgb_imgs):
        if img_rgb is not None:
            indv_savepath = os.path.join(indv_dir, f"region138_cluster{cluster_idx}_proto{i+1}.svg")
            plt.figure(figsize=(4, 5))
            plt.imshow(img_rgb)
            plt.title(f"Prototype #{i+1}")
            plt.axis('off')
            plt.tight_layout()
            plt.close()


In [ ]:
import numpy as np
import pandas as pd
import pickle
import re
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# -------------------- matplotlib style --------------------
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 15
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 20
plt.rcParams['ytick.labelsize'] = 20
plt.rcParams['axes.linewidth'] = 2  # thicker axes/frame

sns.set_style("white")  # seaborn style (no grid by default)

def set_tick_fontsize(ax, size=20):
    for tick in ax.get_yticklabels():
        tick.set_fontsize(size)
    for tick in ax.get_xticklabels():
        tick.set_fontsize(size)

def pval_to_stars(p):
    """p < 0.001 -> ***, p < 0.01 -> **, p < 0.05 -> *, else 'ns'."""
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return 'ns'

# -------------------- load data --------------------
with open("mid_corresponding_patches.pkl", "rb") as f:
    mid_corresponding_patches = pickle.load(f)
with open("late_corresponding_patches.pkl", "rb") as f:
    late_corresponding_patches = pickle.load(f)

# IMPORTANT:
# region_metadata must exist in your environment (as in your original script).
# It should be a dict: region_metadata[region_id] = DataFrame with columns FEATURE_NAME, FEATURE_VALUE

def calculate_class_per_patch(region_ids, class_key='tnm'):
    region_classes = []
    for r in region_ids:
        if r in region_metadata:
            df = region_metadata[r]
            vals = df[df['FEATURE_NAME'] == class_key]['FEATURE_VALUE']
            val = vals.iloc[0] if len(vals) > 0 else np.nan
            region_classes.append(val)
        else:
            region_classes.append(np.nan)
    return region_classes

def split_tnm_labels(labels, return_key='N'):
    clean_exclude = {'-', '', 'nan', 'none', 'unknown'}
    pat = re.compile(r'^(T[0-9A-Za-z]+)(N[0-9A-Za-z]+)(M[0-9A-Za-z]+)$')
    result_labels = []
    for lab in labels:
        if lab is None or not isinstance(lab, str):
            result_labels.append(None)
            continue
        s = lab.strip()
        if s.lower() in clean_exclude:
            result_labels.append(None)
            continue
        s = s.replace(' ', '')
        m = pat.match(s)
        if not m:
            result_labels.append(None)
            continue
        if return_key == 'T':
            result_labels.append(m.group(1))
        elif return_key == 'N':
            result_labels.append(m.group(2))
        elif return_key == 'M':
            result_labels.append(m.group(3))
        else:
            result_labels.append(None)
    return result_labels

def build_long_df(mid_corr, late_corr, class_key='tnm', return_key='N'):
    """
    Returns a long dataframe with columns:
    region_idx, group (Mid/Late), category (e.g., N0/N2 or T2/T4 etc.)
    """
    region_count = len(mid_corr['region_id'])
    rows = []
    for region_idx in range(region_count):
        mid_region_ids = mid_corr['region_id'][region_idx]
        late_region_ids = late_corr['region_id'][region_idx]

        mid_raw_labels = calculate_class_per_patch(mid_region_ids, class_key=class_key)
        late_raw_labels = calculate_class_per_patch(late_region_ids, class_key=class_key)

        mid_labels = split_tnm_labels(mid_raw_labels, return_key=return_key)
        late_labels = split_tnm_labels(late_raw_labels, return_key=return_key)

        for lab in mid_labels:
            if lab is not None and pd.notna(lab):
                rows.append({'region_idx': region_idx, 'group': 'Mid', 'category': lab})
        for lab in late_labels:
            if lab is not None and pd.notna(lab):
                rows.append({'region_idx': region_idx, 'group': 'Late', 'category': lab})

    return pd.DataFrame(rows)

def compute_region_props(df_long, allowed_cats, group_order=('Mid', 'Late')):
    """
    For each region_idx and group, compute proportion of each category among allowed_cats.
    Output columns: region, group, category, prop
    """
    if len(df_long) == 0:
        return pd.DataFrame([])

    df_long = df_long[df_long['category'].isin(allowed_cats)].copy()
    cats_present = [c for c in allowed_cats if c in df_long['category'].unique()]

    prop_rows = []
    for region_idx in df_long['region_idx'].unique():
        for grp in group_order:
            gdf = df_long[(df_long['region_idx'] == region_idx) & (df_long['group'] == grp)]
            total = len(gdf)
            if total == 0:
                continue
            for cat in cats_present:
                cnt = (gdf['category'] == cat).sum()
                prop_rows.append({'region': region_idx, 'group': grp, 'category': cat, 'prop': cnt / total})

    df_prop = pd.DataFrame(prop_rows)
    return df_prop, cats_present

def paired_category_tests(df_prop, categories, group_order=('Mid', 'Late')):
    """
    For each category, do Mann-Whitney U comparing Mid vs Late across regions present in both groups.
    Returns:
      cat_list, pvals_corr, all_x, all_y
    where all_x/all_y are matched series per category.
    """
    cat_list = []
    pvals = []
    all_x = {}
    all_y = {}

    for cat in categories:
        region_x = set(df_prop[(df_prop['category'] == cat) & (df_prop['group'] == group_order[0])]['region'])
        region_y = set(df_prop[(df_prop['category'] == cat) & (df_prop['group'] == group_order[1])]['region'])
        regions_both = region_x.intersection(region_y)

        x_match = df_prop[(df_prop['category'] == cat) &
                          (df_prop['group'] == group_order[0]) &
                          (df_prop['region'].isin(regions_both))]['prop']
        y_match = df_prop[(df_prop['category'] == cat) &
                          (df_prop['group'] == group_order[1]) &
                          (df_prop['region'].isin(regions_both))]['prop']

        cat_list.append(cat)
        all_x[cat] = x_match
        all_y[cat] = y_match

        if (len(x_match) > 1) and (len(y_match) > 1):
            _, pval = mannwhitneyu(x_match, y_match, alternative='two-sided')
            pvals.append(pval)
        else:
            pvals.append(1.0)

    _, pvals_corr, _, _ = multipletests(pvals, method='fdr_bh')
    return cat_list, pvals_corr, all_x, all_y

def plot_violin_with_stars(df_prop, plot_cats, cat_list, pvals_corr, all_x, all_y,
                           xlabel, outpath,
                           palette=None,
                           group_order=('Mid', 'Late'),
                           figsize=(6, 6.5)):
    """
    Seaborn violinplot (no raw datapoints) + significance stars above each category.
    """
    if palette is None:
        palette = {group_order[0]: "#4C72B0", group_order[1]: "#DD8452"}

    plot_df = df_prop[df_prop['category'].isin(plot_cats)].copy()
    if len(plot_df) == 0:
        print(f"No data to plot for {xlabel}.")
        return

    # Build list of corrected pvals aligned with plot_cats
    # FDR-BH adjusted p-value (Benjamini-Hochberg) -> drives significance stars
    plot_pvals_corr = [pvals_corr[list(cat_list).index(c)] for c in plot_cats]

    fig, ax = plt.subplots(figsize=figsize)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

    sns.violinplot(
        data=plot_df,
        x="category",
        y="prop",
        hue="group",
        order=plot_cats,
        hue_order=list(group_order),
        palette=palette,
        cut=0,
        inner="box",      # shows a small box inside violin (still no raw points)
        linewidth=1.5,
        ax=ax
    )

    # Add stars
    ymax_list = []
    for i, (cat, pv) in enumerate(zip(plot_cats, plot_pvals_corr)):

        x_vals = all_x.get(cat, pd.Series([], dtype=float))
        y_vals = all_y.get(cat, pd.Series([], dtype=float))

        ymax = max(
            np.max(x_vals.values) if len(x_vals) > 0 else 0,
            np.max(y_vals.values) if len(y_vals) > 0 else 0
        )

        stars = pval_to_stars(pv)

        if stars != 'ns':  # only draw if significant

            # violin positions: seaborn dodge places them around integer positions
            # Mid is left, Late is right
            x1 = i - 0.2
            x2 = i + 0.2

            line_height = ymax + 0.06
            line_offset = 0.02  # vertical bracket height

            # vertical lines
            ax.plot([x1, x1], [line_height, line_height + line_offset],
                    lw=2, c='black')
            ax.plot([x2, x2], [line_height, line_height + line_offset],
                    lw=2, c='black')

            # horizontal line
            ax.plot([x1, x2],
                    [line_height + line_offset, line_height + line_offset],
                    lw=2, c='black')

            # stars
            ax.text(
                (x1 + x2) / 2,
                line_height + line_offset + 0.01,
                stars,
                ha='center',
                va='bottom',
                fontsize=16,
                color='black',
                fontweight='bold'
            )

    # y-limits
    current_top = ax.get_ylim()[1]
    max_y = max(ymax_list) if len(ymax_list) > 0 else current_top
    ax.set_ylim(0, max(current_top, max_y + 0.18))

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Proportion")

    # clean legend
    ax.legend(title="", frameon=False)

    plt.tight_layout()
    plt.savefig(outpath, bbox_inches="tight")
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.show()

# =========================
#          N PLOT
# =========================
df_long_N = build_long_df(mid_corresponding_patches, late_corresponding_patches,
                          class_key='tnm', return_key='N')

if len(df_long_N) == 0:
    print("No N category data available for plotting.")
else:
    allowed_n_cats = ["N0", "N2"]
    df_prop_N, cats_present_N = compute_region_props(df_long_N, allowed_n_cats, group_order=('Mid', 'Late'))

    if len(df_prop_N) == 0 or len(cats_present_N) == 0:
        print("No region/category group data to plot for N.")
    else:
        cat_list_N, pvals_corr_N, all_x_N, all_y_N = paired_category_tests(df_prop_N, cats_present_N, group_order=('Mid', 'Late'))
        plot_cats_N = [c for c in allowed_n_cats if c in cat_list_N]

        if len(plot_cats_N) == 0:
            print("No N0 or N2 data to plot.")
        else:
            plot_violin_with_stars(
                df_prop=df_prop_N,
                plot_cats=plot_cats_N,
                cat_list=cat_list_N,
                pvals_corr=pvals_corr_N,
                all_x=all_x_N,
                all_y=all_y_N,
                xlabel="Category (N type)",
                outpath=str(OUTPUT_DIR / "N_patient.svg"),
                palette={"Mid": "#4C72B0", "Late": "#DD8452"},
                group_order=('Mid', 'Late'),
                figsize=(6, 6.5)
            )

# =========================
#          T PLOT
# =========================
df_long_T = build_long_df(mid_corresponding_patches, late_corresponding_patches,
                          class_key='tnm', return_key='T')

if len(df_long_T) == 0:
    print("No T category data available for plotting.")
else:
    allowed_t_cats = ["T2", "T4"]
    df_prop_T, cats_present_T = compute_region_props(df_long_T, allowed_t_cats, group_order=('Mid', 'Late'))

    if len(df_prop_T) == 0 or len(cats_present_T) == 0:
        print("No region/category group data to plot for T.")
    else:
        cat_list_T, pvals_corr_T, all_x_T, all_y_T = paired_category_tests(df_prop_T, cats_present_T, group_order=('Mid', 'Late'))
        plot_cats_T = [c for c in allowed_t_cats if c in cat_list_T]

        if len(plot_cats_T) == 0:
            print("No T2 or T4 data to plot.")
        else:
            plot_violin_with_stars(
                df_prop=df_prop_T,
                plot_cats=plot_cats_T,
                cat_list=cat_list_T,
                pvals_corr=pvals_corr_T,
                all_x=all_x_T,
                all_y=all_y_T,
                xlabel="Category (T type)",
                outpath=str(OUTPUT_DIR / "T_patient.svg"),
                palette={"Mid": "#4C72B0", "Late": "#DD8452"},
                group_order=('Mid', 'Late'),
                figsize=(6, 6.5)
            )

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multitest import multipletests

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 20
plt.rcParams['ytick.labelsize'] = 20
plt.rcParams['xtick.labelsize'] = 20
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.linewidth'] = 2  

# Only use these files for this task
with open("late_biomarker_means.pkl", "rb") as f:
    late_biomarker = pickle.load(f)
with open("mid_biomarker_means.pkl", "rb") as f:
    late_biomarker_control = pickle.load(f)

# --- ATTENTION: Ensure he_labels is loaded ---
if "he_labels" not in locals():
    raise RuntimeError("he_labels (KMeans cluster assignments) must be loaded from previous cell!")

# Only use regions (samples) belonging to cluster 3
cluster_group_num = 3
he_labels = np.asarray(he_labels)
group3_indices = np.where(he_labels == cluster_group_num)[0]

if len(late_biomarker) > 0:
    biomarker_list = sorted(late_biomarker[0].keys())
else:
    biomarker_list = []

p_thresh = 0.05

def filter_extreme_values_1d(x, method="iqr", k=1.5, min_n_keep=5):
    x = np.asarray(x, dtype=float)
    mask = np.isfinite(x)
    x0 = x[mask]
    if x0.size == 0:
        return np.array([], dtype=float), np.zeros_like(x, dtype=bool), {"lower": np.nan, "upper": np.nan}
    if method.lower() == "iqr":
        q1 = np.percentile(x0, 25)
        q3 = np.percentile(x0, 75)
        iqr = q3 - q1
        if not np.isfinite(iqr) or iqr == 0:
            lower, upper = np.min(x0), np.max(x0)
        else:
            lower = q1 - k * iqr
            upper = q3 + k * iqr
    elif method.lower() == "mad":
        med = np.median(x0)
        mad = np.median(np.abs(x0 - med))
        if not np.isfinite(mad) or mad == 0:
            lower, upper = np.min(x0), np.max(x0)
        else:
            scale = 1.4826 * mad
            lower = med - k * scale
            upper = med + k * scale
    else:
        raise ValueError("method must be 'iqr' or 'mad'")

    keep0 = (x0 >= lower) & (x0 <= upper)
    if keep0.sum() < min_n_keep and x0.size >= min_n_keep:
        keep0 = np.ones_like(x0, dtype=bool)
        lower, upper = np.min(x0), np.max(x0)
    keep_mask = np.zeros_like(x, dtype=bool)
    idx = np.where(mask)[0]
    keep_mask[idx] = keep0
    return x[keep_mask], keep_mask, {"lower": float(lower), "upper": float(upper)}

outlier_method = "iqr"
outlier_k = 1.5
min_pairs_keep = 5

# --- Only use samples in cluster 3 for statistical test and violin plot ---
all_results_group3 = []
for biomarker in biomarker_list:
    vals_late = []
    vals_ctrl = []
    for region_idx in group3_indices:
        v_late = late_biomarker[region_idx].get(biomarker, np.nan)
        v_ctrl = late_biomarker_control[region_idx].get(biomarker, np.nan)
        try:
            v_late = float(v_late)
            v_ctrl = float(v_ctrl)
        except Exception:
            continue
        if np.isnan(v_late) or np.isnan(v_ctrl):
            continue
        vals_late.append(v_late)
        vals_ctrl.append(v_ctrl)
    vals_late = np.array(vals_late, dtype=float)
    vals_ctrl = np.array(vals_ctrl, dtype=float)
    diff_raw = vals_late - vals_ctrl

    diff, keep_mask, bounds = filter_extreme_values_1d(
        diff_raw, method=outlier_method, k=outlier_k, min_n_keep=min_pairs_keep
    )
    diff = diff_raw
    def safe_wilcoxon(d, alt):
        try:
            if d.size == 0:
                return np.nan, np.nan
            return stats.wilcoxon(d, alternative=alt)
        except Exception:
            return np.nan, np.nan
    stat_greater, pval_greater = safe_wilcoxon(diff, "greater")
    stat_less, pval_less = safe_wilcoxon(diff, "less")
    stat_two, pval_two = safe_wilcoxon(diff, "two-sided")
    result = {
        "biomarker": biomarker,
        "diff_raw": diff_raw,
        "diff": diff,
        "mean_diff": np.nanmean(diff) if diff.size > 0 else np.nan,
        "pval_greater": pval_greater,
        "pval_less": pval_less,
        "pval_two": pval_two,
        "outlier_bounds": bounds,
        "n_pairs_raw": int(diff_raw.size),
        "n_pairs_used": int(diff.size),
    }
    all_results_group3.append(result)

# FDR correction for this group only
pvals_greater = [r['pval_greater'] for r in all_results_group3 if not np.isnan(r['pval_greater'])]
pvals_less = [r['pval_less'] for r in all_results_group3 if not np.isnan(r['pval_less'])]
pvals_two = [r['pval_two'] for r in all_results_group3 if not np.isnan(r['pval_two'])]

idxs_greater = [i for i, r in enumerate(all_results_group3) if not np.isnan(r['pval_greater'])]
idxs_less = [i for i, r in enumerate(all_results_group3) if not np.isnan(r['pval_less'])]
idxs_two = [i for i, r in enumerate(all_results_group3) if not np.isnan(r['pval_two'])]

def fdr_mark(pvals, idxs, results, keyname):
    if len(pvals) > 0:
        reject, pval_corr, _, _ = multipletests(pvals, alpha=p_thresh, method='fdr_bh')
        for arridx, origidx in enumerate(idxs):
            results[origidx][f"{keyname}_fdr"] = pval_corr[arridx]
            results[origidx][f"{keyname}_sig"] = reject[arridx]
    else:
        for idx in idxs:
            results[idx][f"{keyname}_fdr"] = np.nan
            results[idx][f"{keyname}_sig"] = False

fdr_mark(pvals_greater, idxs_greater, all_results_group3, "greater")
fdr_mark(pvals_less, idxs_less, all_results_group3, "less")
fdr_mark(pvals_two, idxs_two, all_results_group3, "two")

print("Wilcoxon signed-rank test results (ONLY cluster 3, after FDR correction).")
print(f"Outlier filtering: method={outlier_method}, k={outlier_k}, min_pairs_keep={min_pairs_keep}\n")

# Show Late > Control results (Top 5 only)
print("Late > Control (alternative: greater, FDR<{}): (Showing Top 5 by mean diff)".format(p_thresh))
greater_sorted = sorted([r for r in all_results_group3 if r['mean_diff'] > 0], key=lambda x: x['mean_diff'], reverse=True)
for r in greater_sorted[:5]:
    mean_diff = r['mean_diff']
    pval = r['pval_greater']
    fdr = r.get('greater_fdr', np.nan)
    sig = r.get("greater_sig", False)
    sig_mark = "*" if sig else ""
    if not np.isnan(pval):
        print(f"  {r['biomarker']}: mean diff = {mean_diff:.3g}, p = {pval:.3g}, FDR = {fdr:.3g} "
              f"{sig_mark} (n_used={r['n_pairs_used']}/{r['n_pairs_raw']})")

print("\n(An asterisk * denotes result is significant after FDR correction.)")

# Show Late < Control results separately (Top 5 only)
print("\nWilcoxon signed-rank test results (ONLY cluster 3, Late < Control, after FDR correction):")
print("Late < Control (alternative: less, FDR<{}): (Showing Top 5 by most negative mean diff)".format(p_thresh))
less_sorted = sorted([r for r in all_results_group3 if r['mean_diff'] < 0], key=lambda x: x['mean_diff'])
for r in less_sorted[:5]:
    mean_diff = r['mean_diff']
    pval = r['pval_less']
    fdr = r.get('less_fdr', np.nan)
    sig = r.get("less_sig", False)
    sig_mark = "*" if sig else ""
    if not np.isnan(pval):
        print(f"  {r['biomarker']}: mean diff = {mean_diff:.3g}, p = {pval:.3g}, FDR = {fdr:.3g} "
              f"{sig_mark} (n_used={r['n_pairs_used']}/{r['n_pairs_raw']})")

print("\n(An asterisk * denotes result is significant after FDR correction.)")

# Optionally, keep the two-sided results if needed (not limited to top 5)
print("\nLate != Control (alternative: two-sided, FDR<{}):".format(p_thresh))
for r in all_results_group3:
    mean_diff = r['mean_diff']
    pval = r['pval_two']
    fdr = r.get('two_fdr', np.nan)
    sig = r.get("two_sig", False)
    sig_mark = "*" if sig else ""
    if not np.isnan(pval):
        print(f"  {r['biomarker']}: mean diff = {mean_diff:.3g}, p = {pval:.3g}, FDR = {fdr:.3g} "
              f"{sig_mark} (n_used={r['n_pairs_used']}/{r['n_pairs_raw']})")

print("\n(An asterisk * denotes result is significant after FDR correction.)")

# --- Violin plot for all biomarkers, only for group 3, with horizontal line at 0, save SVG ---
fig_save_dir = str(OUTPUT_DIR / "cluster3_violinplots")
os.makedirs(fig_save_dir, exist_ok=True)

# Draw greater (Late > Control) and less (Late < Control) results separately

# GREATER: Late > Control (Top 5 only)
plot_data_greater = []
biomarkers_greater = []
greater_top = greater_sorted[:5]
for r in greater_top:
    d = np.asarray(r["diff"], dtype=float)
    d = d[np.isfinite(d)]
    biomarkers_greater.append(r["biomarker"])
    for v in d:
        plot_data_greater.append({"Biomarker": r["biomarker"], "Difference": float(v)})

if len(plot_data_greater) == 0:
    print(f"No finite values to plot for cluster {cluster_group_num} (Late > Control only).")
else:
    df_greater = pd.DataFrame(plot_data_greater)
    mean_order_greater = [r["biomarker"] for r in greater_top]
    fig_w = min(2.5 * len(mean_order_greater) + 2, 20)
    plt.figure(figsize=(fig_w, 6))
    colors = "cornflowerblue"
    ax = sns.violinplot(
        data=df_greater,
        x="Biomarker",
        y="Difference",
        order=mean_order_greater,
        cut=0,
        inner="box",
        color=colors,
        linewidth=2.5
    )
    # Add horizontal line at y=0
    ax.axhline(0, color='gray', linestyle='--', lw=1.7, zorder=0)
    yvals = df_greater["Difference"].values
    yvals = yvals[np.isfinite(yvals)]
    if yvals.size > 0:
        y_min = float(np.min(yvals))
        y_max = float(np.max(yvals))
        y_rng = y_max - y_min
        if y_rng == 0:
            y_rng = max(1e-6, abs(y_max))
        ax.set_ylim(y_min - 0.15 * y_rng, y_max + 0.15 * y_rng)
    label = f"Late-Control (Cluster {cluster_group_num})"
    plt.title(f"{label}: Biomarker Differences (Late > Control Top 5)\nWilcoxon signed-rank, FDR<{p_thresh}")
    plt.ylabel("Difference (Late - Control)")
    plt.xlabel("Biomarker")
    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    plt.tight_layout()
    fname = f"{fig_save_dir}/violinplot_cluster{cluster_group_num}_greater_top5.svg"
    plt.savefig(fname, bbox_inches="tight")
    plt.savefig(fname.replace(".svg", ".png"), dpi=300, bbox_inches="tight")
    print(f"Saved violin plot for cluster {cluster_group_num} (Late > Control, Top 5) to: {fname}")
    plt.show()

# LESS: Late < Control (Top 5 only)
plot_data_less = []
biomarkers_less = []
less_top = less_sorted[:5]
for r in less_top:
    d = np.asarray(r["diff"], dtype=float)
    d = d[np.isfinite(d)]
    biomarkers_less.append(r["biomarker"])
    for v in d:
        plot_data_less.append({"Biomarker": r["biomarker"], "Difference": float(v)})

if len(plot_data_less) == 0:
    print(f"No finite values to plot for cluster {cluster_group_num} (Late < Control only).")
else:
    df_less = pd.DataFrame(plot_data_less)
    mean_order_less = [r["biomarker"] for r in less_top]
    fig_w = min(2.5 * len(mean_order_less) + 2, 20)
    plt.figure(figsize=(fig_w, 6))
    colors = "indianred"
    ax = sns.violinplot(
        data=df_less,
        x="Biomarker",
        y="Difference",
        order=mean_order_less,
        cut=0,
        inner="box",
        color=colors,
        linewidth=2.5
    )
    # Add horizontal line at y=0
    ax.axhline(0, color='gray', linestyle='--', lw=1.7, zorder=0)
    yvals = df_less["Difference"].values
    yvals = yvals[np.isfinite(yvals)]
    if yvals.size > 0:
        y_min = float(np.min(yvals))
        y_max = float(np.max(yvals))
        y_rng = y_max - y_min
        if y_rng == 0:
            y_rng = max(1e-6, abs(y_max))
        ax.set_ylim(y_min - 0.15 * y_rng, y_max + 0.15 * y_rng)
    label = f"Late-Control (Cluster {cluster_group_num})"
    plt.title(f"{label}: Biomarker Differences (Late < Control Top 5)\nWilcoxon signed-rank, FDR<{p_thresh}")
    plt.ylabel("Difference (Late - Control)")
    plt.xlabel("Biomarker")
    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    plt.tight_layout()
    fname = f"{fig_save_dir}/violinplot_cluster{cluster_group_num}_less_top5.svg"
    plt.savefig(fname, bbox_inches="tight")
    plt.savefig(fname.replace(".svg", ".png"), dpi=300, bbox_inches="tight")
    print(f"Saved violin plot for cluster {cluster_group_num} (Late < Control, Top 5) to: {fname}")
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multitest import multipletests
import re

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 15
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 20
plt.rcParams['ytick.labelsize'] = 20
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.linewidth'] = 2  # Increase boundary (axes/frame) line width

#### HEATMAP: biomarker difference per cluster (group-annotated columns)

# ----------- Load biomarker and cluster data ----------- #
with open("late_biomarker_means.pkl", "rb") as f:
    late_biomarker = pickle.load(f)
with open("mid_biomarker_means.pkl", "rb") as f:
    late_biomarker_control = pickle.load(f)

if 'he_labels' not in locals():
    raise RuntimeError("he_labels (KMeans cluster assignments) must be loaded from previous cell!")

if len(late_biomarker) > 0:
    biomarker_list = sorted(late_biomarker[0].keys())
else:
    biomarker_list = []

region_count = len(late_biomarker)
unique_clusters = sorted(np.unique(he_labels))
n_clusters = len(unique_clusters)
cluster_names = ["Cluster {}".format(c) for c in unique_clusters]

# ----------- Define biomarker functional groups (same as previous heatmap) ----------- #
biomarker_groups = {
    "T_cells": ["CD3e", "CD4", "CD8", "CD45", "CD45RA", "CD45RO", "IFNg", "GranzymeB", "FoxP3"],
    "Checkpoints": ["PD1", "PDL1", "LAG3", "VISTA", "IDO1", "ICOS", "CD39"],
    "Myeloid": ["CD11b", "CD11c", "CD14", "CD16", "CD66", "CD68", "CD163", "MPO", "CD36"],
    "B_cells": ["CD19", "CD20", "CD21", "CD79"],
    "Antigen_presentation": ["HLA-ABC", "HLA-DR", "HLA-E"],
    "Vascular_ECM_Stroma": ["CD31", "CD34", "Caveolin1", "CollagenIV", "aSMA", "Podoplanin", "Vimentin"],
    "Tumor_epithelial": ["EpCAM", "ECad", "Keratin8_18", "PanCK", "TP63", "GATA3", "PGP9_5", "Gal3"],
    "Proliferation_survival": ["Ki67", "PCNA", "BCL2"],
    "Other": ["CD38", "CD40", "CD44", "CD141", "Ranck", "RANKL", "DAPI"],
}

group_order = list(biomarker_groups.keys())
# Build ordered biomarker columns (grouped)
ordered_cols = []
col_group = []
for g in group_order:
    markers = [m for m in biomarker_groups[g] if m in biomarker_list]
    for m in markers:
        if m not in ordered_cols:
            ordered_cols.append(m)
            col_group.append(g)

# fallback to original order if not found
if len(ordered_cols) == 0:
    ordered_cols = biomarker_list.copy()
    col_group = ['Other'] * len(biomarker_list)

# mapping biomarker name to column index (convenience for plotting group blocks)
marker_to_col = {m: i for i, m in enumerate(ordered_cols)}

# ------------ Calculate the mean difference per cluster for each biomarker ------------ #
def filter_extreme_values_1d(x, method="iqr", k=1.5, min_n_keep=5):
    x = np.asarray(x, dtype=float)
    mask = np.isfinite(x)
    x0 = x[mask]
    if x0.size == 0:
        return np.array([], dtype=float), np.zeros_like(x, dtype=bool), {"lower": np.nan, "upper": np.nan}

    if method.lower() == "iqr":
        q1 = np.percentile(x0, 25)
        q3 = np.percentile(x0, 75)
        iqr = q3 - q1
        if not np.isfinite(iqr) or iqr == 0:
            lower, upper = np.min(x0), np.max(x0)
        else:
            lower = q1 - k * iqr
            upper = q3 + k * iqr
    elif method.lower() == "mad":
        med = np.median(x0)
        mad = np.median(np.abs(x0 - med))
        if not np.isfinite(mad) or mad == 0:
            lower, upper = np.min(x0), np.max(x0)
        else:
            scale = 1.4826 * mad
            lower = med - k * scale
            upper = med + k * scale
    else:
        raise ValueError("method must be 'iqr' or 'mad'")

    keep0 = (x0 >= lower) & (x0 <= upper)
    if keep0.sum() < min_n_keep and x0.size >= min_n_keep:
        keep0 = np.ones_like(x0, dtype=bool)
        lower, upper = np.min(x0), np.max(x0)

    keep_mask = np.zeros_like(x, dtype=bool)
    idx = np.where(mask)[0]
    keep_mask[idx] = keep0

    return x[keep_mask], keep_mask, {"lower": float(lower), "upper": float(upper)}

outlier_method = "iqr"
outlier_k = 1.5
min_pairs_keep = 5
p_thresh = 0.05

heatmap_matrix = np.zeros((n_clusters, len(ordered_cols)), dtype=np.float32)
signif_matrix = np.zeros((n_clusters, len(ordered_cols)), dtype=bool)  # true if significant
pval_matrix = np.full((n_clusters, len(ordered_cols)), np.nan)

for i_cluster, cluster_idx in enumerate(unique_clusters):
    group_mask = (he_labels == cluster_idx)
    for j_biomarker, biomarker in enumerate(ordered_cols):
        vals_late = []
        vals_ctrl = []
        for region_idx in np.where(group_mask)[0]:
            v_late = late_biomarker[region_idx].get(biomarker, np.nan)
            v_ctrl = late_biomarker_control[region_idx].get(biomarker, np.nan)
            try:
                v_late = float(v_late)
                v_ctrl = float(v_ctrl)
            except Exception:
                continue
            if np.isnan(v_late) or np.isnan(v_ctrl):
                continue
            vals_late.append(v_late)
            vals_ctrl.append(v_ctrl)
        vals_late = np.array(vals_late, dtype=float)
        vals_ctrl = np.array(vals_ctrl, dtype=float)
        diff_raw = vals_late - vals_ctrl
        diff, keep_mask, bounds = filter_extreme_values_1d(
            diff_raw, method=outlier_method, k=outlier_k, min_n_keep=min_pairs_keep
        )
        diff = diff_raw

        # Compute mean difference
        if diff.size == 0:
            mean_diff = np.nan
        else:
            mean_diff = float(np.nanmean(diff))
        heatmap_matrix[i_cluster, j_biomarker] = mean_diff

        # Compute Wilcoxon p-value
        def safe_wilcoxon(d, alt):
            try:
                if d.size == 0:
                    return np.nan, np.nan
                if np.allclose(d, 0):
                    return np.nan, 1.0
                return stats.wilcoxon(d, alternative=alt)
            except Exception:
                return np.nan, np.nan

        _, pval_greater = safe_wilcoxon(diff, "two-sided")
        pval_matrix[i_cluster, j_biomarker] = pval_greater

# FDR correction across all clusters per biomarker ! (flatten, correct, reshape)
flat_pvals = pval_matrix.flatten()
not_nan_mask = ~np.isnan(flat_pvals)
if np.sum(not_nan_mask) > 0:
    reject, pval_fdr, _, _ = multipletests(flat_pvals[not_nan_mask], alpha=p_thresh, method='fdr_bh')
    signif_flat = np.zeros_like(flat_pvals, dtype=bool)
    signif_flat[not_nan_mask] = reject
    signif_matrix = signif_flat.reshape(pval_matrix.shape)
else:
    signif_matrix = np.zeros_like(pval_matrix, dtype=bool)

# ----------- Draw heatmap with clusters as rows and markers as columns, with group annotation ----------- #

fig_w = max(7, 0.8 * len(ordered_cols))
fig_h = max(8, 1.6 * n_clusters)
print(fig_w, fig_h)

plt.figure(figsize=(fig_w, fig_h))
ax = sns.heatmap(
    heatmap_matrix,
    annot=False,
    cmap="vlag",
    center=0,
    xticklabels=ordered_cols,
    yticklabels=cluster_names,
    linewidths=0.7,
    linecolor='#d9d9d9',
    #cbar_kws={"label": "Mean Difference (Late - Control)"}
)
plt.xlabel("Biomarker (grouped)")
plt.ylabel("Cluster")
#plt.title("Mean Difference (Late - Control) per Cluster and Biomarker\n(star = FDR < 0.05, Wilcoxon signed-rank)")

# Rebuild a cluster x biomarker matrix of FDR-BH adjusted p-values (q-values)
# so significance stars are driven by corrected q-values, not raw p-values.
pval_fdr_matrix = np.full_like(pval_matrix, np.nan, dtype=float)
if np.sum(not_nan_mask) > 0:
    pval_fdr_matrix.flat[not_nan_mask] = pval_fdr

# Draw significance stars
# FDR-BH adjusted p-value (Benjamini-Hochberg) drives the thresholds below.
for i in range(n_clusters):
    for j in range(len(ordered_cols)):
        q = pval_fdr_matrix[i, j]
        if np.isnan(q):
            continue
        mark = None
        if q < 0.001:
            mark = "***"
        elif q < 0.01:
            mark = "**"
        elif q < 0.05:
            mark = "*"
        if mark is not None:
            ax.text(
                j + 0.5, i + 0.5, mark,
                color="black", ha="center", va="center",
                fontsize=19, fontweight='bold'
            )

plt.xticks(rotation=90, ha='right')
plt.yticks(rotation=0)
# --- Add biomarker group blocks and labels (as in previous heatmap) ---
# Compute group blocks in column coordinates
group_blocks = []
for g in group_order:
    idxs = [i for i, gg in enumerate(col_group) if gg == g]
    if len(idxs) == 0:
        continue
    left = min(idxs)
    right = max(idxs)
    group_blocks.append((g, left, right))

# Add separators + group labels above blocks
y_bracket = -0.35
y_text = -0.75
for g, left, right in group_blocks:
    # === Thick block boundary ===
    ax.add_line(plt.Line2D([left, right + 1], [0, 0], lw=2.0, color='black'))
    ax.add_line(plt.Line2D([left, right + 1], [n_clusters, n_clusters], lw=2.0, color='black'))
    ax.add_line(plt.Line2D([left, left], [0, n_clusters], lw=2.0, color='black'))
    ax.add_line(plt.Line2D([right + 1, right + 1], [0, n_clusters], lw=2.0, color='black'))

    # === Bracket (括弧) ===
    ax.plot([left, right + 1], [y_bracket, y_bracket],
            color="black", lw=1.3, clip_on=False)
    ax.plot([left, left], [y_bracket, y_bracket + 0.1],
            color="black", lw=1.3, clip_on=False)
    ax.plot([right + 1, right + 1], [y_bracket, y_bracket + 0.1],
            color="black", lw=1.3, clip_on=False)
    # === Group label ===
    x_center = (left + right + 1) / 2
    ax.text(x_center, y_text, g,
            ha='center', va='bottom',
            fontsize=20, fontweight='bold',
            clip_on=False)

plt.tight_layout()

OUT_DIR = str(OUTPUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
fname_png = os.path.join(OUT_DIR, "biomarker_heatmap_per_cluster.png")
fname_svg = os.path.join(OUT_DIR, "biomarker_heatmap_per_cluster.svg")
plt.savefig(fname_svg, bbox_inches="tight")
plt.savefig(fname_png, dpi=300, bbox_inches="tight")
plt.show()
print("[saved]", fname_png)
print("[saved]", fname_svg)

# ------------- NEW: Also draw a clustermap for the difference heatmap ----------

# Create a DataFrame for the clustermap, for better axis labelling
df_heatmap = pd.DataFrame(heatmap_matrix, index=cluster_names, columns=ordered_cols)

# Draw clustermap
cg = sns.clustermap(
    df_heatmap,
    cmap="vlag",
    center=0,
    row_cluster=True,
    col_cluster=True,
    linewidths=0.7,
    linecolor='#d9d9d9',
    figsize=(max(7, 0.65 * len(ordered_cols)), max(8, 1.2 * n_clusters)),
    xticklabels=True,
    yticklabels=True
)
plt.setp(cg.ax_heatmap.xaxis.get_majorticklabels(), rotation=90, ha="right", fontsize=14)
plt.setp(cg.ax_heatmap.yaxis.get_majorticklabels(), rotation=0, ha="right", fontsize=15)

clustermap_png = os.path.join(OUT_DIR, "biomarker_clustermap_per_cluster.png")
clustermap_svg = os.path.join(OUT_DIR, "biomarker_clustermap_per_cluster.svg")
plt.savefig(clustermap_svg, bbox_inches="tight")
plt.savefig(clustermap_png, dpi=300, bbox_inches="tight")
plt.show()
print("[saved clustermap]", clustermap_png)
print("[saved clustermap]", clustermap_svg)

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 13
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

# ----------- Load biomarker and cluster data ----------- #
with open("late_biomarker_means.pkl", "rb") as f:
    late_biomarker = pickle.load(f)
with open("mid_biomarker_means.pkl", "rb") as f:
    late_biomarker_control = pickle.load(f)

if 'he_labels' not in locals():
    raise RuntimeError("he_labels (KMeans cluster assignments) must be loaded from previous cell!")

if len(late_biomarker) > 0:
    biomarker_list = sorted(late_biomarker[0].keys())
else:
    biomarker_list = []

region_count = len(late_biomarker)

# ----------- Only use cluster 3 samples ----------- #
cluster_pick = 3
if cluster_pick not in set(he_labels):
    raise ValueError(f"Cluster {cluster_pick} not present in he_labels!")

selected_indices = np.where(np.array(he_labels) == cluster_pick)[0]
if len(selected_indices) == 0:
    raise ValueError("No samples found in selected cluster.")

# ----------- Build per-patch (region) biomarker difference matrix ----------- #
diff_mat = []
usable_indices = []
for region_idx in selected_indices:
    diff_vec = []
    for biomarker in biomarker_list:
        v_late = late_biomarker[region_idx].get(biomarker, np.nan)
        v_ctrl = late_biomarker_control[region_idx].get(biomarker, np.nan)
        try:
            v_late = float(v_late)
            v_ctrl = float(v_ctrl)
        except Exception:
            v_late = np.nan
            v_ctrl = np.nan
        diff = v_late - v_ctrl
        diff_vec.append(diff)
    if not np.all(np.isnan(diff_vec)):
        diff_mat.append(diff_vec)
        usable_indices.append(region_idx)

diff_mat = np.array(diff_mat, dtype=np.float32)
bm_names = biomarker_list

# Remove columns (biomarkers) with all nan
valid_cols_mask = ~np.all(np.isnan(diff_mat), axis=0)
diff_mat = diff_mat[:, valid_cols_mask]
bm_names = [bm for i, bm in enumerate(biomarker_list) if valid_cols_mask[i]]

# Remove rows with any nan (to allow PCA)
keep_rows = ~np.any(np.isnan(diff_mat), axis=1)
diff_mat_clean = diff_mat[keep_rows]
usable_indices_clean = [usable_indices[i] for i, keep in enumerate(keep_rows) if keep]

if diff_mat_clean.shape[0] < 2:
    raise ValueError("Less than two patches with complete biomarker difference profiles in cluster 3.")

# ----------- PCA -----------
pca = PCA(n_components=2)
diff_pca = pca.fit_transform(diff_mat_clean)

# ----------- Show PCA scatter plot -----------
plt.figure(figsize=(7,6))
plt.scatter(diff_pca[:,0], diff_pca[:,1], c='slateblue', edgecolor='k', s=100, alpha=0.8)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA of Biomarker Differences (Cluster 3)')
# plt.grid(True, linestyle="--", alpha=0.5)  # Remove PCA plot grid
plt.tight_layout()
OUT_DIR = str(OUTPUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
fname_png = os.path.join(OUT_DIR, "cluster3_PC_scatter.png")
fname_svg = os.path.join(OUT_DIR, "cluster3_PC_scatter.svg")
ax = plt.gca()

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.set_yticks([])   
ax.set_xticks([])
plt.savefig(os.path.join(OUT_DIR, "pca_cluster3_biomarker_diff.svg"), bbox_inches="tight")
plt.savefig(os.path.join(OUT_DIR, "pca_cluster3_biomarker_diff.png"), dpi=300, bbox_inches="tight")
plt.show()
print("[saved]", fname_png)
print("[saved]", fname_svg)

# ----------- PC1 contribution barplot ONLY TOP3 POS/NEG ----------- #
pc1_weights = pca.components_[0]
abs_pc1_weights = np.abs(pc1_weights)
pc1_contrib_df = pd.DataFrame({
    "Biomarker": bm_names,
    "PC1_Weight": pc1_weights,
    "Abs_PC1_Weight": abs_pc1_weights
}).set_index("Biomarker")

# Find top 3 positive and top 3 negative contributors
n_show = 5
pc1_sorted = pc1_contrib_df.sort_values("PC1_Weight", ascending=False)
top3_pos = pc1_sorted.head(n_show)
top3_neg = pc1_sorted.tail(n_show).sort_values("PC1_Weight", ascending=True)

# Concat top 3 pos and top 3 neg
bar_df = pd.concat([top3_pos, top3_neg])
bar_order = list(top3_pos.index) + list(top3_neg.index)

plt.figure(figsize=(5, 8))  # Taller, narrower for horizontal bars

# Custom bar colors: less saturated for both positive and negative
def low_sat_green(val, minval, maxval):
    import matplotlib.colors as mcolors
    from matplotlib import cm

    norm = (val - minval) / (maxval - minval + 1e-9)
    if val > 0:
        # Pastel green for positive
        return (0.78, 0.93, 0.78)
    elif val < 0:
        # Pastel light red or salmon
        return (1.0, 0.85, 0.85)
    else:
        return (0.9, 0.9, 0.9)

weights = bar_df.loc[bar_order, "PC1_Weight"].values
minval = bar_df["PC1_Weight"].min()
maxval = bar_df["PC1_Weight"].max()
bar_colors = [low_sat_green(w, minval, maxval) for w in weights]

# --- Rotated barplot: horizontal ---
bars = plt.barh(
    np.arange(len(bar_df)),
    bar_df.loc[bar_order, "PC1_Weight"],
    color=bar_colors, edgecolor='k'
)
# Set fontsize for the x and y tick labels
plt.yticks(
    np.arange(len(bar_df)),
    bar_order,
    fontsize=15  # match plt.rcParams['ytick.labelsize']
)
plt.xticks(fontsize=15)  # match plt.rcParams['xtick.labelsize']
plt.xlabel('PC1 Contribution (Weight)', fontsize=14)
plt.title('Top 3 positive & negative contributors to PC1 (Cluster 3)', fontsize=18)
plt.tight_layout()
fname_png = os.path.join(OUT_DIR, "cluster3_PC1_biomarker_contribution_barplot_top3.png")
fname_svg = os.path.join(OUT_DIR, "cluster3_PC1_biomarker_contribution_barplot_top3.svg")
plt.savefig(fname_svg, bbox_inches="tight")
plt.savefig(fname_png, dpi=300, bbox_inches="tight")
plt.show()
print("[saved]", fname_png)
print("[saved]", fname_svg)



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

# This section computes the mean value for each biomarker channel across the (256, 256, 40) patch arrays
# for all usable_indices_clean patches, and computes correlations with PCA axes.

# Assume query_patches['patch_data'] is shape [num_patches, 256, 256, 40] (or similar), and
# query_patches['channels'] provides biomarker names.
# query_patches may be dict or DataFrame, adapt as needed.

he3_pca = diff_pca

if isinstance(query_patches, dict):
    # Expect shape [num_patches, 256, 256, 40]
    patch_data = query_patches['codex']
    biomarkers = query_patches['channels'][0]
else:
    # DataFrame-like fallback
    patch_data = query_patches['codex'].values
    biomarkers = query_patches['channels'][0].values

# Only keep usable indices
patch_data_filtered = np.array(patch_data)[usable_indices_clean]  # shape [num_used, 256, 256, 40]
biomarker_list = list(biomarkers)

# Compute per-patch channelwise mean values: [num_patches, 40]
patch_means = np.nanmean(patch_data_filtered, axis=(2, 3)) # averages over height and width

# Now, compute summary stats and correlations with the PCA axes (already filtered by usable_indices_clean)
from scipy.stats import pearsonr
pc1 = he3_pca[:, 0]
pc2 = he3_pca[:, 1]

summary = []
for b_idx, b in enumerate(biomarker_list):
    vals = patch_means[:, b_idx]
    mask = np.isfinite(vals) & np.isfinite(pc1) & np.isfinite(pc2)
    vals_masked = vals[mask]
    p1 = pc1[mask]
    p2 = pc2[mask]
    mean_val = np.nanmean(vals_masked) if len(vals_masked) > 0 else np.nan
    pcc_1 = pearsonr(vals_masked, p1)[0] if len(vals_masked) > 1 else np.nan
    pcc_2 = pearsonr(vals_masked, p2)[0] if len(vals_masked) > 1 else np.nan
    summary.append({
        "biomarker": b,
        "mean_patch_mean_value": mean_val,
        "pearson_corr_pc1": pcc_1,
        "pearson_corr_pc2": pcc_2,
        "n_patches": int(len(vals_masked))
    })

summary_df = pd.DataFrame(summary)
print(summary_df)

OUT_DIR = str(OUTPUT_DIR / "he_label3_pc_biomarker_patchmean")
os.makedirs(OUT_DIR, exist_ok=True)
csv_path = os.path.join(OUT_DIR, "biomarker_pca_patchmean_stats.csv")
print("Saved:", csv_path)

plot_df = summary_df.set_index("biomarker")[["pearson_corr_pc1", "pearson_corr_pc2"]]
plt.figure(figsize=(3.5, max(4.0, 0.36 * plot_df.shape[0])))
sns.heatmap(plot_df, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("PCC of Patch Mean Biomarker Values with PC1/PC2")
plt.tight_layout()
out_png = os.path.join(OUT_DIR, "biomarker_pcc_with_pc1pc2_patchmean.png")
out_svg = os.path.join(OUT_DIR, "biomarker_pcc_with_pc1pc2_patchmean.svg")
plt.savefig(out_svg, bbox_inches="tight")
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()
print("[saved]", out_png)
print("[saved]", out_svg)

print("Mean biomarker values (per-patch mean of 256x256) for label 3 patches:")
display(summary_df[["biomarker", "mean_patch_mean_value"]])


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from scipy.stats import pearsonr, linregress

plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 15
plt.rcParams['xtick.labelsize'] = 15
plt.rcParams['ytick.labelsize'] = 15
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

# --- MODIFY: Select a list of biomarkers of interest to plot PC1 vs their mean patch values in a panel format ---

# Example list, change as needed:
biomarker_names = [
    "GATA3",
    "Vimentin",
    "ICOS",
    "PDL1",
    "CD3e",
    'CD11b'
]
# List can be changed/expanded as needed

def find_biomarker_idx(name, biomarker_list):
    for i, b in enumerate(biomarker_list):
        if b.replace(" ", "").lower() == name.replace(" ", "").lower():
            return i
    return None

# Optionally, support alternative names per biomarker
alternative_names = {
    "CD68": ["CD68"],
    "Keratin8_18": ["Keratin8_18", "Keratin8", "Keratin8-18"],
    "CD3": ["CD3"],
    "CD8": ["CD8"],
    "CD20": ["CD20"],
    # add more as needed
}

he3_pca = diff_pca      # shape [num_used, n_components]
pc1 = he3_pca[:, 0]

# Find indices for all biomarkers
biomarker_indices = []
biomarker_found_names = []
for bm in biomarker_names:
    idx = None
    for alt_name in alternative_names.get(bm, [bm]):
        idx = find_biomarker_idx(alt_name, biomarker_list)
        if idx is not None:
            break
    if idx is not None:
        biomarker_indices.append(idx)
        biomarker_found_names.append(biomarker_list[idx])
    else:
        print(f"Biomarker '{bm}' not found in biomarker_list.")
        biomarker_indices.append(None)
        biomarker_found_names.append(f"{bm} (not found)")

n_biomarkers = len(biomarker_names)
n_cols = min(3, n_biomarkers)
n_rows = int(np.ceil(n_biomarkers / n_cols))

fig, axs = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4.5 * n_rows), squeeze=False)

# Save results for each biomarker (scatter and statistics)
results = []
for i, (bm, idx, found_name) in enumerate(zip(biomarker_names, biomarker_indices, biomarker_found_names)):
    r, c = divmod(i, n_cols)
    ax = axs[r][c]
    if idx is not None:
        bm_vals = patch_means[:, idx]
        mask = np.isfinite(pc1) & np.isfinite(bm_vals)
        x = pc1[mask]
        y = bm_vals[mask]
        r_value, pval = pearsonr(x, y) if len(x) > 1 else (np.nan, np.nan)
        fit = linregress(x, y) if len(x) > 1 else None
        ax.scatter(x, y, alpha=0.7)
        if fit:
            ax.plot(np.sort(x), fit.intercept + fit.slope * np.sort(x), color='red', lw=2)
        ax.set_xlabel('PC1', fontweight='bold')
        ax.set_ylabel(f"{found_name} mean patch value", fontweight='bold')
        ax.set_title(f'PC1 vs {found_name} (patch mean)', fontweight='bold')
        ax.text(0.05, 0.95, f'PCC: {r_value:.2f}\np={pval:.2e}',
                ha='left', va='top', transform=ax.transAxes,
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'), fontweight='bold')
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)
        results.append({
            "biomarker": found_name,
            "pearson_corr_pc1": r_value,
            "pearson_pval_pc1": pval,
            "n_patches": int(len(y))
        })
    else:
        ax.set_axis_off()
        ax.text(0.5, 0.5, f"{bm}\n(not found)", ha='center', va='center', color='red', fontsize=14, fontweight='bold')
        results.append({
            "biomarker": bm,
            "pearson_corr_pc1": np.nan,
            "pearson_pval_pc1": np.nan,
            "n_patches": 0
        })

# Hide unused axes if any
for i in range(n_biomarkers, n_rows * n_cols):
    r, c = divmod(i, n_cols)
    axs[r][c].set_axis_off()

plt.tight_layout()
scatter_dir = str(OUTPUT_DIR / "he_label3_pc_biomarker_summary_panel")
os.makedirs(scatter_dir, exist_ok=True)
scatter_png = os.path.join(scatter_dir, "scatter_pc1_multi_biomarker_patchmean.png")
scatter_svg = os.path.join(scatter_dir, "scatter_pc1_multi_biomarker_patchmean.svg")
plt.savefig(scatter_svg, bbox_inches="tight")
plt.savefig(scatter_png, dpi=300, bbox_inches="tight")
plt.show()
print(f"[saved] {scatter_png}")
print(f"[saved] {scatter_svg}")

# Save results/statistics to CSV
res_df = pd.DataFrame(results)
csv_path = os.path.join(scatter_dir, "biomarker_pc1_patchmean_stats.csv")
print(f"[saved stats] {csv_path}")
